# 07b · 이토 공식 (Itô's formula)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Øksendal §4.1 (1차원 Itô 공식, Thm 4.1.2), §4.2 (다차원·부분적분) · Shreve II §4.4 (Itô–Doeblin 공식, 예제 4.4.7–4.4.8) · G&S 4e §13.9 · Lawler 2e Ch. 9 (Itô's formula 절) |
| 선수 노트북 | 07a (Itô 적분·등거리성), 06b (이차변동 $[B]_t=t$), 06e (기하 브라운 운동) |
| 예상 소요 | 100분 |
| 상태 | draft |

07a에서 $\int_0^T B\,dB=(B_T^2-T)/2$ 라는, 보통 미적분과 다른 답을 얻었다. 이 노트북은 그 "$-T/2$"가 우연이 아니라 **연쇄법칙(chain rule)의 확률 버전**에서 나오는 체계적인 항임을 보인다: $f(B_t)$ 의 변화는 $f'(B)\,dB$ 만이 아니라 $\frac12f''(B)\,dt$ 를 더 가진다. 이유는 단 하나 — 06b에서 본 $\sum\Delta B_i^2\to t$, 즉 "$dB^2=dt$" 다. 이 항을 완전 증명으로 확인하고, 같은 경로를 점점 거칠게 만들며 잔차가 $n^{-1/2}$ 로 사라지는 것을 보고, 곱셈 규칙·생성원·기하 브라운 운동의 $-\sigma^2/2$ 까지 한 줄로 꿰어 본다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths, coarsen
from spkit.sde import euler_maruyama
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("07b")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_STEPS = 1000
T = 1.0
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_STEPS={N_STEPS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (07a) $\int_0^T B\,dB$ 의 Itô 값과 Stratonovich 값은? 두 값의 차이는 어디서 오는가?

<details><summary>정답</summary>

Itô: $(B_T^2-T)/2$, Stratonovich: $B_T^2/2$. 차이 $T/2$ 는 이차변동 $\sum\Delta B_i^2\to T$ 의 절반이다 (왼쪽 끝점 대신 중점에서 평가하면 $\frac12\sum\Delta B_i^2$ 만큼 더 들어온다).

</details>

**Q2.** (07a) Itô 등거리성(isometry)을 진술하고, 어느 가정이 교차항을 없애는지 말하라.

<details><summary>정답</summary>

$E\big[(\int_0^TH\,dB)^2\big]=E\int_0^TH_t^2\,dt$. **적합성**(adaptedness): $H_{t_i}$ 가 $\mathcal F_{t_i}$-가측이라 미래 증분 $\Delta B_i$ 와 독립이고, 따라서 $i<j$ 교차항 $E[H_{t_i}\Delta B_iH_{t_j}\Delta B_j]=E[H_{t_i}\Delta B_iH_{t_j}]\,E[\Delta B_j]=0$.

</details>

**Q3.** (07a) 왼쪽 끝점 리만합 $L_n=\sum B_{t_i}\Delta B_i$ 의 오차 RMS 는 $n$ 에 어떻게 의존하는가?

<details><summary>정답</summary>

$T/\sqrt{2n}$, 즉 $n^{-1/2}$ (log-log 기울기 $-1/2$). 오늘의 잔차 실험(step 2)도 같은 기울기를 낸다 — 같은 원인($\sum(\Delta B_i^2-\Delta t)$ 의 분산 $2T\Delta t$)이다.

</details>

**Q4.** (06e) 기하 브라운 운동의 해 $S_t=S_0\exp\big((\mu-\sigma^2/2)t+\sigma B_t\big)$ 에서 $\log S_t$ 의 분포는?

<details><summary>정답</summary>

$\log S_t\sim N\big(\log S_0+(\mu-\sigma^2/2)t,\ \sigma^2t\big)$ (로그정규). 06e 에서는 이 $-\sigma^2/2$ 를 "$E[S_t]=S_0e^{\mu t}$ 가 되도록 맞춘 상수"로 받아들였다. 오늘 그 정체를 밝힌다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **Itô 과정**(Itô process). $X_t=X_0+\int_0^ta_s\,ds+\int_0^tb_s\,dB_s$, 약식으로 $dX=a\,dt+b\,dB$. $a,b$ 는 적합 과정이고 $\int_0^t|a_s|ds<\infty$ a.s., $E\int_0^tb_s^2\,ds<\infty$ (07a 의 Itô 적분이 정의되는 조건).
- **곱셈표**(multiplication table, heuristic). $dt\cdot dt=0$, $dt\cdot dB=0$, $dB\cdot dB=dt$. 마지막 규칙은 06b 의 이차변동 $[B]_t=t$ 를 "미분형"으로 쓴 것이다 — 증명에서는 $\sum g(B_{t_i})(\Delta B_i^2-\Delta t)\to0$ 이라는 보조정리로 나타난다.
- **Itô 공식 (BM).** $f\in C^2$ 이면
  $$f(B_t)=f(B_0)+\int_0^tf'(B_s)\,dB_s+\frac12\int_0^tf''(B_s)\,ds,\qquad\text{약식: } df(B)=f'(B)\,dB+\tfrac12f''(B)\,dt.$$
- **Itô 공식 (Itô 과정, 시간 의존).** $f\in C^{1,2}$ ($t$ 에 대해 1번, $x$ 에 대해 2번 연속 미분가능) 이면
  $$df(t,X_t)=\Big(f_t+a\,f_x+\tfrac12b^2f_{xx}\Big)dt+b\,f_x\,dB.$$
- **곱셈 규칙**(product rule). $d(X_tY_t)=X_t\,dY_t+Y_t\,dX_t+dX_t\,dY_t$, 여기서 $dX\,dY=b^Xb^Y\,dt$ (같은 BM 으로 구동) 또는 $0$ (독립 BM 으로 구동).
- **생성원**(generator)과 **Dynkin 공식.** BM 의 생성원은 $\mathcal Lf=\frac12f''$ 이고, Itô 적분의 평균이 0 이므로
  $$E[f(B_t)]=f(0)+\int_0^tE[\mathcal Lf(B_s)]\,ds,\qquad \frac{d}{dt}E[f(B_t)]=E[\mathcal Lf(B_t)].$$

### 2.2 정리 1 — Itô 공식 (BM; $f\in C^2$, $f''$ 유계·연속) — Øksendal §4.1 Thm 4.1.2, Shreve II §4.4.1–4.4.3

$t>0$ 를 고정하고 격자 $t_i=i\Delta t$, $\Delta t=t/n$, $\Delta B_i=B_{t_{i+1}}-B_{t_i}$ 라 하자. 이산 잔차

$$R_n:=f(B_t)-f(0)-\sum_{i<n}f'(B_{t_i})\Delta B_i-\frac12\sum_{i<n}f''(B_{t_i})\Delta t$$

는 $n\to\infty$ 에서 확률수렴으로 0 이 되고, 따라서 $f(B_t)-f(0)=\int_0^tf'(B_s)dB_s+\frac12\int_0^tf''(B_s)ds$ a.s.

가정이 왜 필요한가:
- **$f\in C^2$**: 2차 테일러 전개가 필요하고, 2차 항이 $\Delta B_i^2\approx\Delta t$ 로 살아남기 때문이다. $C^1$ 만으로는 $|x|$ 처럼 $\frac12\int f''$ 대신 **국소시간**(local time) 항이 나타난다 (E3, Tanaka 공식).
- **$f''$ 유계** (또는 $f''$ 연속 + 국소화): 2차 항 $\sum f''(B_{t_i})(\Delta B_i^2-\Delta t)$ 의 분산이 $2\Delta t^2\sum E[f''(B_{t_i})^2]\le2\|f''\|_\infty^2\,t\,\Delta t\to0$ 이 되게 하려고. 일반 $C^2$ 는 정지시간 $\tau_K=\inf\{s:|B_s|\ge K\}$ 로 국소화하면 된다 (경로가 연속이라 $\tau_K\uparrow\infty$).
- **이차변동 $\sum\Delta B_i^2\to t$ ($L^2$, 06b)**: "$dB^2=dt$" 규칙의 근거이자 $\frac12f''$ 항의 출처.
- **적합성**: $f'(B_{t_i})$ 를 **왼쪽 끝점**에서 평가해야 $\sum f'(B_{t_i})\Delta B_i\to\int f'(B)dB$ (Itô 적분, 07a). 중점을 쓰면 Stratonovich 적분이 되고 $\frac12f''$ 항이 사라진다.

<details><summary>증명</summary>

**0. 테일러 전개.** $f\in C^2$ 이므로 각 $i$ 에 대해 어떤 $\xi_i$ ($B_{t_i}$ 와 $B_{t_{i+1}}$ 사이) 가 있어

$$f(B_{t_{i+1}})-f(B_{t_i})=f'(B_{t_i})\Delta B_i+\tfrac12f''(\xi_i)\Delta B_i^2
=f'(B_{t_i})\Delta B_i+\tfrac12f''(B_{t_i})\Delta B_i^2+\rho_i,$$

$\rho_i=\frac12\big(f''(\xi_i)-f''(B_{t_i})\big)\Delta B_i^2$, 따라서 $|\rho_i|\le\frac12\,\omega(|\Delta B_i|)\,\Delta B_i^2$. 여기서 $\omega$ 는 $f''$ 의 연속률(modulus of continuity) — $f''$ 이 유계·균등연속이면 전역 $\omega$ 를, 아니면 경로가 머무는 구간 $[-K,K]$ 위의 $\omega_K$ 를 쓴다(아래 4단계 참고). $i$ 에 대해 더하면 망원합이 되어

$$f(B_t)-f(0)=\underbrace{\sum f'(B_{t_i})\Delta B_i}_{(1)}+\underbrace{\tfrac12\sum f''(B_{t_i})\Delta t}_{(2)}+\underbrace{\tfrac12\sum f''(B_{t_i})(\Delta B_i^2-\Delta t)}_{(3)=\frac12Z_n}+\underbrace{\sum\rho_i}_{(4)}.$$

즉 $R_n=\frac12Z_n+\sum\rho_i$ 이다. 네 조각을 하나씩 처리한다.

**(1) 확률적분 항.** $f''$ 유계이면 $f'$ 은 선형 성장이라 $E\int_0^tf'(B_s)^2ds<\infty$ 이고, $s\mapsto f'(B_s)$ 는 연속 적합 과정이다. 07a 의 Itô 적분 구성(단순 과정 근사 + 등거리성)에 의해 $\sum f'(B_{t_i})\Delta B_i\to\int_0^tf'(B_s)dB_s$ ($L^2$).

**(2) 드리프트 항.** $s\mapsto f''(B_s)$ 는 (경로별로) 연속 함수이므로 왼쪽 끝점 리만합은 $\frac12\int_0^tf''(B_s)ds$ 로 경로별 수렴한다.

**(3) 핵심 보조정리.** $g$ 유계 가측이면 $Z_n=\sum_{i<n}g(B_{t_i})(\Delta B_i^2-\Delta t)\to0$ ($L^2$).

*증명.* 제곱을 전개하면 $E[Z_n^2]=\sum_i E[g(B_{t_i})^2(\Delta B_i^2-\Delta t)^2]+2\sum_{i<j}E[g(B_{t_i})(\Delta B_i^2-\Delta t)\,g(B_{t_j})(\Delta B_j^2-\Delta t)]$.
교차항($i<j$): 앞의 세 인자는 $\mathcal F_{t_j}$-가측이고 $\Delta B_j^2-\Delta t$ 는 $\mathcal F_{t_j}$ 와 독립이며 평균 0 이므로 (07a 의 등거리성 증명과 같은 논리) 조건부 기댓값을 취하면 0.
대각항: $g(B_{t_i})$ 와 $\Delta B_i$ 가 독립이고 $E[(\Delta B_i^2-\Delta t)^2]=\mathrm{Var}(\Delta B_i^2)=3\Delta t^2-\Delta t^2=2\Delta t^2$ (정규분포 4차 모멘트). 따라서

$$E[Z_n^2]=2\Delta t^2\sum_{i<n}E[g(B_{t_i})^2]\le2\|g\|_\infty^2\,n\,\Delta t^2=2\|g\|_\infty^2\,t\,\Delta t\to0.\quad\square$$

$g=f''$ 로 두면 (3) $\to0$ ($L^2$).

**(4) 나머지 항.** $\sum_i|\rho_i|\le\frac12\max_i\omega(|\Delta B_i|)\cdot\sum_i\Delta B_i^2$. 경로가 $[0,t]$ 에서 균등연속이므로 $\max_i|\Delta B_i|\to0$ a.s., 따라서 $\max_i\omega(|\Delta B_i|)\to0$ a.s.; 한편 $\sum\Delta B_i^2\to t$ (확률수렴, 06b). 곱은 $0\cdot t=0$ 으로 확률수렴한다. ($f''$ 이 균등연속이 아닌 일반 $C^2$ 의 경우: 사건 $\{\max_{s\le t}|B_s|\le K\}$ 위에서 $[-K,K]$ 의 연속률 $\omega_K$ 로 같은 논증을 하고, $K\to\infty$ 에서 이 사건의 확률이 1 로 가므로 확률수렴은 그대로 성립한다.)

**결론.** $R_n=\frac12Z_n+\sum\rho_i\to0$ (확률수렴). 한편 (1),(2) 의 극한이 각각 $\int f'(B)dB$, $\frac12\int f''(B)ds$ 이므로 $f(B_t)-f(0)$ 은 이 두 극한의 합과 a.s. 같다. $\blacksquare$

**부산물 (step 2 에서 관측).** $R_n\approx\frac12Z_n$ 이므로

$$E[R_n^2]\approx\tfrac14E[Z_n^2]=\tfrac12\Delta t^2\sum_iE[f''(B_{t_i})^2]\approx\frac{\Delta t}{2}\int_0^tE[f''(B_s)^2]\,ds=O(1/n).$$

RMS 잔차의 log-log 기울기는 $-1/2$. 예: $f=e^x$, $t=1$: $E[e^{2B_s}]=e^{2s}$ 이므로 $E[R_n^2]\approx\frac{1}{2n}\cdot\frac{e^2-1}{2}=\frac{e^2-1}{4n}$; $f=x^3$: $f''=6x$, $E[36B_s^2]=36s$ 이므로 $E[R_n^2]\approx\frac{1}{2n}\cdot18=\frac9n$.

</details>

### 2.3 정리 2 — Itô 과정에 대한 Itô 공식과 곱셈 규칙 — Øksendal §4.2 Thm 4.2.1, Shreve II §4.4.4

$dX=a\,dt+b\,dB$, $f\in C^{1,2}$ 이면 $df(t,X)=(f_t+af_x+\frac12b^2f_{xx})\,dt+b\,f_x\,dB$. 특히 $d(XY)=X\,dY+Y\,dX+dX\,dY$ 이고, 독립 BM $B,W$ 에 대해 $d(B_tW_t)=B\,dW+W\,dB$ (교차변동 0). 이산 교차변동 $C_n=\sum_{i<n}\Delta B_i\Delta W_i$ 는 $E[C_n]=0$, $E[C_n^2]=T^2/n$ 으로 **정확히** 계산된다.

가정이 왜 필요한가:
- **$b$ 적합·$L^2$**: $\int b\,f_x\,dB$ 가 Itô 적분으로 정의되려면.
- **$f\in C^{1,2}$**: $t$ 에 대해 1차, $x$ 에 대해 2차 전개가 필요하다. $dt^2$, $dt\,dB$ 항은 $o(\Delta t)$ 라서 합해도 사라진다 (곱셈표의 앞 두 규칙).
- **독립 BM 의 교차변동이 0**: $\Delta B_i\Delta W_i$ 는 평균 0, 분산 $\Delta t^2$ 인 독립 항이므로 $\mathrm{Var}(C_n)=n\Delta t^2=T^2/n\to0$. 같은 BM 이면 $\sum\Delta B_i^2\to T$ (06b) — 이것이 $dX\,dY=b^Xb^Y\,dt$ 의 출처.

*증명 스케치.* 정리 1 과 같은 테일러 전개를 $(t,x)$ 두 변수에 대해 하고 $\Delta X_i=a_i\Delta t+b_i\Delta B_i$ 를 대입한다. $\Delta X_i^2=b_i^2\Delta B_i^2+2a_ib_i\Delta t\Delta B_i+a_i^2\Delta t^2$ 에서 첫 항만 $O(\Delta t)$ 로 살아남고($\to b^2dt$), 나머지는 $O(\Delta t^{3/2}),O(\Delta t^2)$ 라 합해도 0 으로 간다. $\Delta t\cdot\Delta X_i$ 항도 마찬가지. 곱셈 규칙은 $f(x,y)=xy$ 에 2차원 Itô 공식을 적용한 것이다: $f_x=y$, $f_y=x$, $f_{xx}=f_{yy}=0$, $f_{xy}=1$ 이므로 $d(XY)=Y\,dX+X\,dY+dX\,dY$. 자세한 것은 Øksendal §4.2 Thm 4.2.1. $\square$

### 2.4 정리 3 — Dynkin 공식 (BM) — Øksendal §4.1, G&S 4e §13.9

$f\in C^2$, $f',f''$ 유계(또는 적절한 적분가능성)이면 $E[f(B_t)]=f(0)+\frac12\int_0^tE[f''(B_s)]\,ds$, 즉 $\frac{d}{dt}E[f(B_t)]=E[\mathcal Lf(B_t)]$, $\mathcal L=\frac12\partial_x^2$.

가정이 왜 필요한가:
- $\int f'(B)dB$ 가 **진짜 마팅게일**(평균 0) 이어야 한다: $E\int_0^tf'(B_s)^2ds<\infty$ 면 07a 의 마팅게일 성질로 보장된다. 그렇지 않으면 국소 마팅게일에 그쳐 평균이 0 이 아닐 수 있다 (함정 5).

*증명 스케치.* Itô 공식(정리 1) 양변의 기댓값을 취한다. 확률적분 항은 평균 0, 드리프트 항은 Fubini 로 $\frac12\int_0^tE[f''(B_s)]ds$. 예:
- $f=x^4$: $f''=12x^2$, $E[B_t^4]=\frac12\int_0^t12\,E[B_s^2]ds=6\int_0^ts\,ds=3t^2$ (정규분포 4차 모멘트 $3\sigma^4$ 와 일치).
- $f=e^x$: $m(t):=E[e^{B_t}]$ 는 $m'=\frac12m$, $m(0)=1$ → $m(t)=e^{t/2}$.
- $f=\cos x$: $m'=-\frac12m$ → $m(t)=e^{-t/2}$. $\square$

### 2.5 직관

보통 미적분의 연쇄법칙 $df=f'\,dx$ 는 $dx^2$ 이 무시할 만큼 작다는 데 기댄다. BM 은 $dB\sim\sqrt{dt}$ 라 $dB^2\sim dt$ 가 1차 항과 **같은 크기**로 살아남고, 이것이 $\frac12f''\,dt$ 라는 "보정 드리프트"가 된다. 볼록한 $f$ ($f''>0$: $x^2$, $e^x$) 는 요동 자체가 평균을 밀어 올린다 — 젠센 부등식의 미분 버전이다: $E[e^{B_t}]=e^{t/2}>1=e^{E[B_t]}$. 오목한 $f$ ($\cos$ 의 0 근처, $\log$) 는 평균을 끌어내린다: 이것이 GBM 의 $-\sigma^2/2$ 다. 06e 에서 "지수 안에 $-\sigma^2/2$ 가 있다"고 외웠던 것이 07b 에서는 "$\log$ 의 2차 도함수가 $-1/x^2$ 이기 때문"으로 설명된다:

$$d\log S=\frac{1}{S}\,dS+\frac12\Big(-\frac1{S^2}\Big)(dS)^2=\frac{\mu S\,dt+\sigma S\,dB}{S}-\frac12\frac{\sigma^2S^2\,dt}{S^2}=\Big(\mu-\frac{\sigma^2}2\Big)dt+\sigma\,dB.$$

### 2.6 함정

1. $d(B_t^2)=2B_t\,dB_t$ 가 **아니다**: $+dt$ 가 붙는다 (07a 의 $(B_T^2-T)/2$ 가 바로 이것).
2. Itô 공식은 $C^2$ 를 요구한다. $f(x)=|x|$ 에 형식적으로 적용하면 $|B_t|=\int\mathrm{sgn}(B)dB$ (평균 0) 이 되어 $E|B_t|=\sqrt{2t/\pi}>0$ 과 모순 — 빠진 항이 국소시간 $L_t^0$ (Tanaka 공식, E3).
3. $\frac12f''$ 항은 "작은 보정"이 아니다: $E[e^{B_1}]$ 에서 1 과 $e^{1/2}=1.65$ 의 차이 **전부**가 이 항에서 온다.
4. Stratonovich 적분($\circ\,dB$)을 쓰면 연쇄법칙이 보통 형태로 돌아오지만 마팅게일 성질을 잃는다. 이 과정에서는 Itô 만 쓴다.
5. Dynkin 공식의 확률적분 항이 평균 0 이려면 진짜 마팅게일이어야 한다. $f$ 가 너무 빨리 자라면 ($e^{B^2}$ 등) $E\int f'(B)^2ds=\infty$ 가 되어 실패할 수 있다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — "$dB^2=dt$" 가 왜 0 이 아닌지 06b 의 이차변동 그림을 떠올리며 2문장으로, 그 결과로 $e^{B_t}$ 의 평균이 1 이 아니라 $e^{t/2}$ 가 되는 이유를 1문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

아래 값을 **손으로** 계산해 적어 보세요 (Itô 공식 + 기댓값이면 충분합니다). 시뮬레이션 뒤 5절의 표에서 자기 예측과 비교합니다.

In [ ]:
predictions = {
    # E[B_1^4] 은? (힌트: d(B^4) = 4B^3 dB + 6B^2 dt 의 기댓값을 적분)
    "E_B4": None,
    # E[e^{B_1}] 은? (힌트: m(t)=E[e^{B_t}] 의 미분방정식)
    "E_expB": None,
    # f=cos 일 때 Itô 보정항 (1/2)∫_0^1 f''(B_s) ds = -(1/2)∫_0^1 cos(B_s) ds 의 **평균**은?
    "drift_cos": None,
    # GBM dS = 0.05 S dt + 0.5 S dB, S_0=1, T=1 에서 E[log S_1] 은? (순진한 답 μT=0.05 와 비교)
    "E_logS": None,
    # 같은 GBM 에서 E[S_1] 은?
    "E_S": None,
    # 독립 BM B, W 의 이산 교차변동 C_n = Σ ΔB_i ΔW_i (T=1, n=1000) 에 대해 E[C_n^2] 은?
    "msq_cross_n1000": None,
}

## 4. Simulate — 시뮬레이션

### Step 1 — 경로별 Itô 공식

경로 하나하나에 대해 이산 Itô 공식의 세 조각 `lhs = f(B_T)-f(0)`, `stoch = Σ f'(B_{t_i}) ΔB_i`, `drift = ½ Σ f''(B_{t_i}) Δt` 를 계산하고 잔차 `res = lhs - stoch - drift` 를 본다. 잔차가 **경로별로** 작아야 한다 — 평균만 맞는 것이 아니다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def ito_residual(f, f1, f2, t, B):
    """격자 t 와 경로 B (n_paths, n+1) 에 대해 (lhs, stoch, drift, res) 를 반환.
    왼쪽 끝점 평가, dt = t[1]-t[0]."""
    dt = t[1] - t[0]
    dB = np.diff(B, axis=1)
    Bl = B[:, :-1]                      # 왼쪽 끝점 B_{t_i}
    lhs = f(B[:, -1]) - f(B[:, 0])
    stoch = (f1(Bl) * dB).sum(axis=1)   # Σ f'(B_{t_i}) ΔB_i  → ∫ f'(B) dB
    drift = 0.5 * (f2(Bl) * dt).sum(axis=1)   # ½ Σ f''(B_{t_i}) Δt → ½ ∫ f''(B) ds
    return lhs, stoch, drift, lhs - stoch - drift

# (f, f', f'')
fs = {
    "x2":  (lambda x: x**2, lambda x: 2 * x, lambda x: 2.0 + 0 * x),
    "x3":  (lambda x: x**3, lambda x: 3 * x**2, lambda x: 6 * x),
    "x4":  (lambda x: x**4, lambda x: 4 * x**3, lambda x: 12 * x**2),
    "exp": (np.exp, np.exp, np.exp),
    "cos": (np.cos, lambda x: -np.sin(x), lambda x: -np.cos(x)),
}

먼저 경로를 만들고, 한 경로에서 $f=\exp$ 의 running 값을 보자: $\frac12\int f''$ 를 **빼먹은** 곡선이 어떻게 벌어지는지.

In [ ]:
t, B = brownian_paths(N_PATHS, N_STEPS, T, rng)
dt = t[1] - t[0]
dB = np.diff(B, axis=1)
Bl, BT = B[:, :-1], B[:, -1]

f, f1, f2 = fs["exp"]
k = 0                                    # 경로 하나
stoch_inc = f1(Bl[k]) * dB[k]            # f'(B_{t_i}) ΔB_i
drift_inc = 0.5 * f2(Bl[k]) * dt         # ½ f''(B_{t_i}) Δt
run_true = f(B[k])
run_noc = f(0.0) + np.concatenate([[0.0], np.cumsum(stoch_inc)])
run_ito = f(0.0) + np.concatenate([[0.0], np.cumsum(stoch_inc + drift_inc)])

fig, ax = plt.subplots()
ax.plot(t, run_true, lw=2, label="$f(B_t) = e^{B_t}$")
ax.plot(t, run_noc, "--", label="$f(0) + \\sum f'(B)\\,\\Delta B$  (no correction)")
ax.plot(t, run_ito, ":", lw=2, label="$f(0) + \\sum f'(B)\\,\\Delta B + \\frac{1}{2}\\sum f''(B)\\,\\Delta t$  (Itô)")
ax.set(xlabel="t", ylabel="value", title="Itô's formula on one path, f = exp")
ax.legend(loc="lower left")
plt.show()

Itô 곡선은 참값 위에 겹쳐 보이고, 보정 없는 곡선은 아래로 처진다 — 처지는 양이 정확히 $\frac12\int_0^te^{B_s}ds>0$ 이다. 이제 모든 경로·모든 $f$ 에 대해 잔차의 RMS 를 잰다.

In [ ]:
parts = {}   # name -> (lhs, stoch, drift, res)
print(f"n = {N_STEPS}, paths = {N_PATHS}")
print(f"{'f':>4} | {'RMS lhs':>8} | {'RMS res':>8} | {'res/lhs':>8}")
for name, (f, f1, f2) in fs.items():
    lhs, stoch, drift, res = ito_residual(f, f1, f2, t, B)
    parts[name] = (lhs, stoch, drift, res)
    rms_lhs, rms_res = np.sqrt((lhs**2).mean()), np.sqrt((res**2).mean())
    print(f"{name:>4} | {rms_lhs:8.4f} | {rms_res:8.4f} | {rms_res / rms_lhs:8.4f}")

잔차의 RMS 가 좌변 RMS 의 2–3 % 수준이다. 이것이 "0 으로 간다"는 주장인지는 step 2 에서 $n$ 을 바꿔 확인한다.

### Step 2 — 잔차의 수렴률 (같은 경로를 coarsen)

**같은 경로**를 `factor` 배 성기게 보면 격자 $n=1000/\text{factor}$ 의 이산 Itô 공식이 된다. 잔차 RMS 가 $n^{-1/2}$ 로 줄어드는지, 증명의 부산물 $\sqrt{(e^2-1)/(4n)}$ ($f=\exp$), $\sqrt{9/n}$ ($f=x^3$) 과 맞는지 본다.

In [ ]:
factors = [100, 50, 20, 10, 5, 2, 1]
ns = np.array([N_STEPS // fac for fac in factors])
rms = {"exp": [], "x3": []}
for fac in factors:
    tk, Bk = coarsen(t, B, fac)
    for name in rms:
        f, f1, f2 = fs[name]
        res = ito_residual(f, f1, f2, tk, Bk)[3]
        rms[name].append(np.sqrt((res**2).mean()))
rms = {k: np.array(v) for k, v in rms.items()}

ref = {"exp": np.sqrt((np.e**2 - 1) / (4 * ns)), "x3": np.sqrt(9 / ns)}
slopes = {k: np.polyfit(np.log(ns), np.log(v), 1)[0] for k, v in rms.items()}
print(f"{'n':>5} | {'RMS exp':>8} {'ref':>8} | {'RMS x3':>8} {'ref':>8}")
for i, n in enumerate(ns):
    print(f"{n:5d} | {rms['exp'][i]:8.4f} {ref['exp'][i]:8.4f} | {rms['x3'][i]:8.4f} {ref['x3'][i]:8.4f}")
print("log-log slopes:", {k: round(v, 3) for k, v in slopes.items()}, "(expected -0.5)")

In [ ]:
fig, ax = plt.subplots()
for name, marker in [("exp", "o"), ("x3", "s")]:
    ax.loglog(ns, rms[name], marker, label=f"f = {name}, slope {slopes[name]:.2f}")
    ax.loglog(ns, ref[name], "--", color="gray", lw=1)
ax.loglog(ns, rms["exp"][0] * (ns / ns[0]) ** -0.5, ":", color="k", label="reference slope -1/2")
ax.set(xlabel="n (grid points)", ylabel="RMS residual R_n",
       title="RMS residual of the discrete Itô formula vs n\n(dashed: leading-order sqrt((e²-1)/(4n)), sqrt(9/n))")
ax.legend()
plt.show()

기울기 $-1/2$ 는 07a 의 $L_n$ 오차와 같은 기울기다. 이유도 같다: 잔차의 주된 부분은 $\frac12\sum f''(B_{t_i})(\Delta B_i^2-\Delta t)$ 이고 그 분산이 $\propto\Delta t$ 다. 점선 참조 곡선은 **선행 차수** 근사이므로 작은 $n$ 에서 (고차 테일러 항 때문에) 몇 % 위에 관측점이 놓이는 것이 정상이다.

### Step 3 — 생성원과 Dynkin 공식

같은 경로에서 $f\in\{x^4,\exp,\cos\}$ 에 대해 (a) $E[f(B_1)]$ 과 닫힌 형식 $3,\ e^{1/2},\ e^{-1/2}$, (b) 보정항 평균 $E[\frac12\int f''(B)ds]$ 과 $E[f(B_1)]-f(0)$, (c) 확률적분 평균 $\approx0$ 을 비교한다. (b)+(c) 가 (a) 를 만든다 — 즉 평균의 변화는 **전부** 보정항에서 온다.

In [ ]:
exact_Ef = {"x4": 3.0, "exp": np.exp(0.5), "cos": np.exp(-0.5)}   # 정리 3 의 세 예
for name in ["x4", "exp", "cos"]:
    f = fs[name][0]
    lhs, stoch, drift, res = parts[name]
    print(f"f = {name:>3}:  E[f(B_1)] = {mc_estimate(f(BT))}  vs exact {exact_Ef[name]:.6f}")
    print(f"          E[½∫f''(B)ds] = {mc_estimate(drift)}  vs E[f(B_1)]-f(0) = {exact_Ef[name] - f(0.0):.6f}")
    print(f"          E[∫f'(B)dB]   = {mc_estimate(stoch)}  vs 0")

$x^4$ 의 SE 가 유난히 크다: $B_1^4$ 의 분산은 $E[B^8]-9=105-9=96$ 이라 표준편차 $\approx9.8$ 이다. 반면 보정항 $6\int B^2ds$ 는 경로에 대해 평균을 취한 양이라 분산이 훨씬 작다 — Dynkin 공식으로 계산하는 편이 MC 로도 유리하다는 뜻이다.

In [ ]:
ks = np.arange(100, N_STEPS + 1, 100)          # t_k = 0.1, ..., 1.0
tk = t[ks]
tt = np.linspace(0, 1, 200)
exact_curve = {"x4": 3 * tt**2, "exp": np.exp(tt / 2), "cos": np.exp(-tt / 2)}
labels = {"x4": "$E[B_t^4] = 3t^2$", "exp": "$E[e^{B_t}] = e^{t/2}$", "cos": "$E[\\cos B_t] = e^{-t/2}$"}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, name in zip(axes, ["x4", "exp", "cos"]):
    f = fs[name][0]
    vals = f(B[:, ks])
    m, se = vals.mean(0), vals.std(0, ddof=1) / np.sqrt(N_PATHS)
    ax.errorbar(tk, m, yerr=2 * se, fmt="o", ms=4, capsize=3, label="MC ± 2 SE")
    ax.plot(tt, exact_curve[name], "-", label=labels[name])
    ax.set(xlabel="t", title=f"f = {name}")
    ax.legend(loc="upper left" if name != "cos" else "upper right")
fig.suptitle("E[f(B_t)] from MC vs generator solution", y=1.02)
plt.show()

$\frac{d}{dt}E[f(B_t)]=\frac12E[f''(B_t)]$ 가 세 곡선 모두를 만든다: 볼록한 $x^4,\exp$ 는 위로, $0$ 근처에서 오목한 $\cos$ 는 아래로.

### Step 4 — 곱셈 규칙과 교차변동

독립 BM $B,W$ 에 대해 $d(BW)=B\,dW+W\,dB+dB\,dW$. 이산 버전 $B_TW_T-\sum(B_{t_i}\Delta W_i+W_{t_i}\Delta B_i)=\sum\Delta B_i\Delta W_i$ 는 항등식(망원합)이라 **수치적으로 정확히** 성립하고, 우변의 교차변동 $C_n$ 이 $0$ 으로 가는지가 문제다.

In [ ]:
_, W = brownian_paths(N_PATHS, N_STEPS, T, rng)       # B 와 독립인 BM
dW = np.diff(W, axis=1)
Wl, WT = W[:, :-1], W[:, -1]

lhs_prod = BT * WT
rhs_prod = (Bl * dW + Wl * dB).sum(axis=1)             # Σ (B dW + W dB)
cross = (dB * dW).sum(axis=1)                           # C_n = Σ ΔB ΔW
qv_same = (dB * dB).sum(axis=1)                         # Σ ΔB² (같은 BM, 06b)

assert np.allclose(lhs_prod - rhs_prod, cross)          # 망원합 항등식 (결정적)
print("max |lhs - rhs - cross| =", np.abs(lhs_prod - rhs_prod - cross).max())
print("E[C_n]   =", f"{mc_estimate(cross):.6f}", " vs 0")
print("E[C_n^2] =", f"{mc_estimate(cross**2):.6f}", f" vs T^2/n = {T**2 / N_STEPS:.6f}")
print("Σ ΔB²    =", mc_estimate(qv_same), " vs T = 1 (same BM)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(cross, bins=50, density=True)
axes[0].axvline(0, color="k", lw=1)
axes[0].set(xlabel="Σ ΔB ΔW", title=f"Cross-variation of independent BMs (n={N_STEPS})\nsd = T/√n = {T / np.sqrt(N_STEPS):.4f}")
axes[1].hist(qv_same, bins=50, density=True)
axes[1].axvline(T, color="k", lw=1)
axes[1].set(xlabel="Σ ΔB²", title=f"Quadratic variation of one BM (n={N_STEPS})\nsd = T·√(2/n) = {T * np.sqrt(2 / N_STEPS):.4f}")
plt.show()

두 히스토그램의 축 눈금을 비교하라: 교차변동은 $0$ 주위 폭 $T/\sqrt n$, 이차변동은 $T$ 주위 폭 $T\sqrt{2/n}$. 둘 다 $n\to\infty$ 에서 상수로 붕괴하지만 **다른 상수**($0$ 과 $T$)로 — 곱셈표 $dB\,dW=0$, $dB\,dB=dt$ 가 이것이다.

### Step 5 — Itô 과정 버전: GBM 에 $\log$ 를 적용

$dS=\mu S\,dt+\sigma S\,dB$ 를 오일러–마루야마(Euler–Maruyama, EM)로 풀고 $\log S_1$ 의 평균을 본다. 2.5절의 계산대로 $d\log S=(\mu-\sigma^2/2)dt+\sigma\,dB$ 이므로 $E[\log S_1]=\mu-\sigma^2/2$ 이지, 순진한 $\mu$ 가 아니다. EM 의 약수렴 편향은 $O(\Delta t)$ 인데, $\log(1+\mu\Delta t+\sigma\Delta B)$ 를 4차까지 전개해 더하면 선행항이 $T\Delta t\,(-\mu^2/2+\mu\sigma^2-3\sigma^4/4)$ 로 이 매개변수에서는 $10^{-5}$ 급 — SE 의 1 % 수준이라 무시해도 된다 (07c 에서 정량화).

In [ ]:
mu, sigma, S0 = 0.05, 0.5, 1.0
_, S = euler_maruyama(lambda t, x: mu * x, lambda t, x: sigma * x,
                      S0, T, N_STEPS, rng, n_paths=N_PATHS)
S1 = S[:, -1]
logS1 = np.log(S1)
m_log = np.log(S0) + (mu - sigma**2 / 2) * T          # E[log S_T]
print("E[log S_1] =", mc_estimate(logS1), f" vs μ-σ²/2 = {m_log:.4f}  (naive μT = {mu * T:.4f})")
print("E[S_1]     =", mc_estimate(S1), f" vs S0·e^{{μT}} = {S0 * np.exp(mu * T):.6f}")
z_naive = (mc_estimate(logS1).mean - mu * T) / mc_estimate(logS1).se
print(f"naive μT is {abs(z_naive):.1f} SE away from the MC mean")

In [ ]:
x = np.linspace(logS1.min(), logS1.max(), 300)
fig, ax = plt.subplots()
plot_hist_vs_pdf(logS1, x, norm.pdf(x, m_log, sigma * np.sqrt(T)), ax=ax,
                 label_pdf="N(μ-σ²/2, σ²) density")
ax.axvline(mu * T, color="C3", ls="--", label=f"naive μT = {mu * T:.3f}")
ax.axvline(m_log, color="k", ls=":", lw=2, label=f"μ-σ²/2 = {m_log:.3f}")
ax.set(xlabel="$\\log S_1$", title="$\\log S_1$ from Euler–Maruyama GBM (μ=0.05, σ=0.5)")
ax.legend()
plt.show()

$E[S_1]=e^{\mu}$ 은 06e 그대로인데 $E[\log S_1]$ 은 $\mu$ 보다 $\sigma^2/2$ 만큼 작다. 두 사실은 모순이 아니라 젠센 부등식 $E[\log S]<\log E[S]$ 의 정확한 양이 $-\sigma^2T/2$ 라는 뜻이다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다 (정리 3 의 세 ODE 해, $T^2/n$, GBM 의 $\mu-\sigma^2/2$). 보정항 세 행은 왼쪽 끝점 합 $\frac12\sum_{i<n}f''(B_{t_i})\Delta t$ 의 **이산** 기댓값과 비교한다 — $E[f''(B_{t_i})]$ 가 각각 $12t_i$, $e^{t_i/2}$, $-e^{-t_i/2}$ 이므로 등차·등비급수 합으로 정확히 계산된다 ($x^4$: $3(1-1/n)$). 연속 극한 $E[f(B_1)]-f(0)$ 과의 차이는 코드에서 출력한다.

In [ ]:
n = N_STEPS
exact = {
    "E_B4": 3.0 * T**2,                        # 3t²
    "drift_x4": 0.5 * dt * (12 * t[:-1]).sum(),             # ½Δt Σ 12 t_i = 3(1-1/n)
    "E_expB": np.exp(T / 2),                   # e^{t/2}
    "drift_exp": 0.5 * dt * np.exp(t[:-1] / 2).sum(),       # ½Δt Σ e^{t_i/2} (등비급수)
    "E_cosB": np.exp(-T / 2),                  # e^{-t/2}
    "drift_cos": -0.5 * dt * np.exp(-t[:-1] / 2).sum(),     # -½Δt Σ e^{-t_i/2}
    "stoch_exp": 0.0,                          # Itô 적분의 평균
    "msq_cross_n1000": T**2 / n,               # Var(C_n) = n Δt²
    "E_logS": np.log(S0) + (mu - sigma**2 / 2) * T,
    "E_S": S0 * np.exp(mu * T),
    "msq_res_exp": (np.e**2 - 1) / (4 * n),    # 선행 차수 근사 (assert 하지 않음)
}
print("discrete vs continuous correction mean:")
print(f"  x4 : {exact['drift_x4']:.6f} vs 3          (3(1-1/n) = {3 * (1 - 1 / n):.6f})")
print(f"  exp: {exact['drift_exp']:.6f} vs {exact['E_expB'] - 1:.6f}")
print(f"  cos: {exact['drift_cos']:.6f} vs {exact['E_cosB'] - 1:.6f}")

In [ ]:
est = {
    "E_B4": mc_estimate(BT**4),
    "drift_x4": mc_estimate(parts["x4"][2]),
    "E_expB": mc_estimate(np.exp(BT)),
    "drift_exp": mc_estimate(parts["exp"][2]),
    "E_cosB": mc_estimate(np.cos(BT)),
    "drift_cos": mc_estimate(parts["cos"][2]),
    "stoch_exp": mc_estimate(parts["exp"][1]),
    "msq_cross_n1000": mc_estimate(cross**2),
    "E_logS": mc_estimate(logS1),
    "E_S": mc_estimate(S1),
    "msq_res_exp": mc_estimate(parts["exp"][3] ** 2),
}

In [ ]:
names = {
    "E_B4": "E[B_1^4]",
    "drift_x4": "Itô correction mean, f=x^4: E[6∫B² ds] (discrete exact)",
    "E_expB": "E[e^{B_1}]",
    "drift_exp": "Itô correction mean, f=exp: E[½∫e^B ds] (discrete exact)",
    "E_cosB": "E[cos B_1]",
    "drift_cos": "Itô correction mean, f=cos: E[-½∫cos B ds] (discrete exact)",
    "stoch_exp": "E[∫e^B dB] (stochastic integral)",
    "msq_cross_n1000": f"E[C_n^2], cross-variation of indep. BMs (n={n})",
    "E_logS": "E[log S_1] (GBM via EM)",
    "E_S": "E[S_1] (GBM via EM)",
    "msq_res_exp": f"E[R_n^2], f=exp, n={n} (leading order only, not asserted)",
}
rows = [{"name": names[k], "predicted": predictions.get(k), "estimate": est[k], "exact": exact[k]}
        for k in names]
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **E[B_1^4]**, **E[e^{B_1}]**, **E[cos B_1]**: Dynkin 공식(정리 3)의 ODE 해가 틀렸거나 경로 생성이 잘못됨. ($x^4$ 행은 SE 가 커서 판정이 느슨하다.)
- **Itô correction mean (x^4, exp, cos)**: $\frac12\int f''(B)ds$ 의 평균이 $E[f(B_1)]-f(0)$ 과 다르면 확률적분의 평균이 0 이 아니라는 뜻 — 적합성(왼쪽 끝점)이 깨졌을 때 생긴다.
- **E[∫e^B dB]**: 0 에서 벗어나면 마팅게일 성질 위반(오른쪽/중점 평가 등).
- **E[C_n^2]**: $T^2/n$ 과 다르면 $B,W$ 가 독립이 아니거나 증분 분산이 $\Delta t$ 가 아님 (표는 소수 4자리로 반올림하므로 step 4 의 6자리 출력을 보라).
- **E[log S_1]**: $\mu-\sigma^2/2$ 대신 $\mu$ 근처면 $\log$ 의 Itô 보정을 빠뜨린 것; 둘 다 아니면 EM 편향이 SE 보다 큰 것(격자를 촘촘히).
- **E[S_1]**: $S_0e^{\mu T}$ 와 다르면 EM 구현 또는 드리프트 계수 오류.
- **E[R_n^2]**: 참조값은 선행 차수라 몇 % 차이는 정상 — "investigate" 가 떠도 $O(n^{-1})$ 규모 자체가 맞으면 된다 (step 2 의 기울기가 진짜 검증).

In [ ]:
assert_close(est["E_B4"], exact["E_B4"], k=4, name="E[B^4]")
assert_close(est["drift_x4"], exact["drift_x4"], k=4, name="Ito correction x^4")
assert_close(est["E_expB"], exact["E_expB"], k=4, name="E[exp B]")
assert_close(est["drift_exp"], exact["drift_exp"], k=4, name="Ito correction exp")
assert_close(est["E_cosB"], exact["E_cosB"], k=4, name="E[cos B]")
assert_close(est["drift_cos"], exact["drift_cos"], k=4, name="Ito correction cos")
assert_close(est["stoch_exp"], exact["stoch_exp"], k=4, name="mean of stochastic integral")
assert_close(est["msq_cross_n1000"], exact["msq_cross_n1000"], k=4, name="cross variation msq")
assert_close(est["E_logS"], exact["E_logS"], k=4, name="E[log S]")
assert_close(est["E_S"], exact["E_S"], k=4, name="E[S]")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

(a) $f(t,x)=e^{\theta x-\theta^2t/2}$ 에 시간 의존 Itô 공식을 적용해 $dt$ 항이 사라짐을 보이고, 이것이 06d 의 지수 마팅게일임을 설명하라.
(b) Itô + 기댓값으로 $E[B_t^4]=3t^2$ 을 유도하고 $t=2$ 에서 값을 구하라.
(c) $E[B_t^6]$ 을 같은 방법으로 구하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $f_t=-\frac{\theta^2}{2}f$, $f_x=\theta f$, $f_{xx}=\theta^2f$. $a=0,b=1$ 인 Itô 공식: $df=(f_t+\frac12f_{xx})dt+f_x\,dB=\big(-\frac{\theta^2}2f+\frac{\theta^2}2f\big)dt+\theta f\,dB=\theta f\,dB$. 드리프트가 없는 Itô 적분이므로 ($E\int f^2ds<\infty$ 확인 가능) 마팅게일 — 06d 에서 $E[e^{\theta B_t}]=e^{\theta^2t/2}$ 로 직접 확인했던 지수 마팅게일 $M_t=e^{\theta B_t-\theta^2t/2}$ 이다. $-\theta^2t/2$ 는 정확히 $\frac12f_{xx}$ 를 상쇄하려고 붙는다.

(b) $d(B^4)=4B^3\,dB+\frac12\cdot12B^2\,dt=4B^3\,dB+6B^2\,dt$. 기댓값: $E[B_t^4]=6\int_0^tE[B_s^2]ds=6\int_0^ts\,ds=3t^2$. $t=2$: $E[B_2^4]=12$.

(c) $d(B^6)=6B^5\,dB+\frac12\cdot30B^4\,dt=6B^5\,dB+15B^4\,dt$. $E[B_t^6]=15\int_0^t3s^2\,ds=15t^3$ ($t=1$: 15, 정규분포 6차 모멘트 $15\sigma^6$ 와 일치). 일반적으로 $E[B_t^{2k}]=(2k-1)!!\,t^k$ 가 귀납적으로 나온다.

</details>

### E2 유도 후 시뮬레이션 검증

$Y_t=e^{B_t}$ 의 SDE $dY=Y\,dB+\frac12Y\,dt$ 를 유도하고, $Y^2=e^{2B}$ 에도 Itô 를 적용해 $E[Y_t^2]=e^{2t}$, 따라서 $\mathrm{Var}(Y_t)=e^{2t}-e^t$ 임을 보여라. $t=1$ 에서 MC 로 검증하라 (상대 SE 가 5% 정도로 큰 이유도 설명).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$f=e^x$ 는 $f'=f''=f$ 이므로 $dY=e^B\,dB+\frac12e^B\,dt=Y\,dB+\frac12Y\,dt$; 기댓값을 취하면 $m'=\frac12m$, $E[Y_t]=e^{t/2}$. $g=e^{2x}$ 는 $g'=2g$, $g''=4g$ 이므로 $d(e^{2B})=2e^{2B}\,dB+2e^{2B}\,dt$ → $E[e^{2B_t}]$ 는 $m'=2m$ 을 풀어 $e^{2t}$. 따라서 $\mathrm{Var}(Y_t)=e^{2t}-e^{t}$, $t=1$: $e^2-e=4.6708$.

```python
Y = np.exp(BT)
var_mc = Y.var(ddof=1)
se_normal = var_mc * np.sqrt(2 / (N_PATHS - 1))     # 정규 근사 SE (과소평가)
print(f"Var(e^B_1) MC = {var_mc:.3f} ± {se_normal:.3f}  vs exact {np.e**2 - np.e:.4f}")
print("E[e^{2B_1}] =", mc_estimate(Y**2), f" vs e^2 = {np.e**2:.4f}")
```

상대 오차가 큰 이유: $Y$ 는 로그정규라 꼬리가 무겁다. 표본분산의 분산은 $(\mu_4-\mathrm{Var}(Y)^2)/N$ 인데 4차 중심모멘트 $\mu_4$ 는 $E[Y^4]=E[e^{4B}]=e^8\approx2981$ 이 지배해 $\mathrm{Var}(Y)^2\approx22$ 의 100배가 넘는다. 그래서 정규 근사 SE $\mathrm{Var}\sqrt{2/(N-1)}$ (20 000 경로에서 상대 1 %) 는 실제 산포(상대 약 7.5 %)를 크게 과소평가하고, 5–8 % 의 차이가 보통이다. $E[e^{2B_1}]$ 의 MC 평균도 마찬가지다: 그 SE 는 $\mathrm{Var}(Y^2)=E[Y^4]-E[Y^2]^2=e^8-e^4$ 로 정해져 상대 SE 가 20 000 경로에서 5 % 안팎이다 (`mc_estimate` 출력의 SE 로 확인).

</details>

### E3 가정을 깨보기

$f(x)=|x|$ 는 $C^2$ 가 아니다. 형식적으로 $f'=\mathrm{sgn}$, $f''=0$ (a.e.) 로 두고 Itô 공식을 적용하면 $|B_1|=\int_0^1\mathrm{sgn}(B_s)dB_s$ 가 되어 평균이 0 이어야 한다. step 1 의 경로 `B` 로 $E|B_1|$ 과 $E[\sum\mathrm{sgn}(B_{t_i})\Delta B_i]$ 를 각각 추정해 모순을 확인하고, 차이 $|B_1|-\int\mathrm{sgn}(B)dB$ 의 평균이 $\sqrt{2/\pi}$ 임을 관찰하라 (Tanaka 공식: 빠진 항은 국소시간 $L_1^0$).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

```python
absB = np.abs(BT)
sgn_int = (np.sign(Bl) * dB).sum(axis=1)             # Σ sgn(B_{t_i}) ΔB_i  (적합 → 평균 0)
print("E|B_1|            =", mc_estimate(absB), f" vs sqrt(2/π) = {np.sqrt(2 / np.pi):.5f}")
print("E[∫ sgn(B) dB]    =", mc_estimate(sgn_int), " vs 0")
print("E[|B_1| - ∫sgn dB] =", mc_estimate(absB - sgn_int), f" vs E[L_1^0] = {np.sqrt(2 / np.pi):.5f}")
```

$E|B_1|=\sqrt{2/\pi}=0.79788$ (반정규분포). 확률적분 항은 적합 피적분함수의 Itô 적분이므로 평균 0 — MC 도 0 근처. 차이의 평균 $E[L_1^0]=\sqrt{2/\pi}$ (Tanaka: $|B_t|=\int_0^t\mathrm{sgn}(B_s)dB_s+L_t^0$, 그리고 $E[L_t^0]=E|B_t|$). 교훈: $f''$ 이 존재하지 않는 점 $0$ 에 BM 이 머무는 시간은 0 이지만, "이차변동 × 곡률"의 기여는 델타함수처럼 살아남아 (형식적으로 $f''=2\delta_0$) $\frac12\int2\delta_0(B_s)ds=L_t^0$ 이 된다. Itô 공식의 $C^2$ 가정은 장식이 아니다.

</details>

## 7. 정리

1. **Itô 공식**: $df(B)=f'(B)\,dB+\frac12f''(B)\,dt$ — 2차 테일러 항이 $dB^2=dt$ 때문에 살아남는다. 완전 증명의 핵심은 보조정리 $\sum g(B_{t_i})(\Delta B_i^2-\Delta t)\to0$ ($L^2$, 분산 $2\|g\|^2t\Delta t$).
2. 이산 잔차는 $O(n^{-1/2})$ 로 사라진다 — 같은 경로를 coarsen 해서 log-log 기울기 $-1/2$ 와 선행항 $\sqrt{(e^2-1)/(4n)}$ 을 확인했다.
3. **Itô 과정 버전** $df=(f_t+af_x+\frac12b^2f_{xx})dt+bf_x\,dB$ 와 **곱셈 규칙** $d(XY)=X\,dY+Y\,dX+dX\,dY$; 독립 BM 의 교차변동은 $E[C_n^2]=T^2/n$ 으로 정확히 사라진다.
4. 확률적분 항의 평균이 0 이므로 $\frac{d}{dt}E[f(B_t)]=\frac12E[f''(B_t)]$ (**Dynkin**): $E[B_1^4]=3$, $E[e^{B_1}]=e^{1/2}$, $E[\cos B_1]=e^{-1/2}$.
5. GBM 의 $-\sigma^2/2$ 는 $\log$ 의 오목성($f''=-1/x^2$) 이 만든 Itô 보정이다: $E[\log S_1]=\mu-\sigma^2/2$, 그러면서도 $E[S_1]=e^{\mu}$.

**Trap 카드.** Q: $dB^2=dt$ 는 왜 "무시할 수 있는 2차 항"이 아닌가?
→ A: $[0,t]$ 의 이차변동 $\sum\Delta B_i^2$ 이 0 이 아니라 $t$ 로 수렴하기 때문(06b). 그래서 2차 테일러 항이 $\frac12\int f''(B)ds$ 로 누적되고, $E[e^{B_1}]=e^{1/2}$ 처럼 평균을 바꾼다.

**다음 노트북: 07c (오일러–마루야마·밀스타인).** Itô 공식은 SDE 의 해를 손으로 구할 때(07d OU, 07e GBM) 쓰지만 대부분의 SDE 는 닫힌 해가 없다. 07c 에서 수치 해법을 배우고, 밀스타인의 보정항 $\frac12bb'(\Delta B^2-\Delta t)$ 가 바로 오늘 본 $dB^2=dt$ (보조정리의 $\Delta B_i^2-\Delta t$) 에서 나옴을 본다.

`log/progress.md` 한 줄 템플릿:
`- [ ] 07b 이토 공식 — 날짜: ____ / 소요: ___분 / 예측 적중: _/6 / 메모: ____`